<a href="https://colab.research.google.com/github/lindajia9/IDXExchange-ds58/blob/main/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [62]:
files = [
    "CRMLSSold202502.csv",
    "CRMLSSold202503.csv",
    "CRMLSSold202504.csv",
    "CRMLSSold202505.csv",
    "CRMLSSold202506.csv",
    "CRMLSSold202507.csv",
    "CRMLSSold202508.csv",
    "CRMLSSold202509.csv",
    "CRMLSSold202510.csv",
    "CRMLSSold202511.csv",
    "CRMLSSold202512.csv",
    "CRMLSSold202601.csv",
    "CRMLSSold202602.csv",
    "CRMLSSold202603.csv",
    "CRMLSSold202604.csv",
    "CRMLSSold202605.csv"
]

dfs = [pd.read_csv(file) for file in files]

df = pd.concat(dfs, ignore_index=True)

/tmp/ipykernel_1233/3919767068.py:20: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]
/tmp/ipykernel_1233/3919767068.py:20: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  dfs = [pd.read_csv(file) for file in files]


In [63]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]

### Inspect Missing Values

In [64]:
print(len(df))
print("\nMissing values after preprocessing:")
print(df.isnull().sum()[df.isnull().sum() > 0])

173338

Missing values after preprocessing:
BuyerAgentAOR                     3678
ListAgentAOR                        32
Flooring                         61908
ViewYN                           15657
WaterfrontYN                    173251
                                 ...  
HighSchoolDistrict               46727
PostalCode                           2
AssociationFee                   50505
LotSizeSquareFeet                 3004
MiddleOrJuniorSchoolDistrict    173338
Length: 62, dtype: int64


In [65]:
# check how many missing values each boolean column has
cols = [
    "ViewYN",
    "PoolPrivateYN",
    "AttachedGarageYN",
    "FireplaceYN",
    "NewConstructionYN"
]

print(df[cols].isna().sum())

ViewYN               15657
PoolPrivateYN        13539
AttachedGarageYN     20872
FireplaceYN            127
NewConstructionYN    13138
dtype: int64



### Handling missing values (before split)








In [66]:
# Drop columns where more than 70% of values are missing
threshold = 0.7
df = df.loc[:, df.isnull().mean() < threshold]

In [67]:
# drop rows that have missing values in the ClosePrice column
df = df.dropna(subset=["ClosePrice"])

# Remove observations where ClosePrice is less than or equal to 0
df = df[df['ClosePrice'] > 0]

In [68]:
# Remove duplicate rows
df.drop_duplicates(inplace=True)

In [69]:
# Remove observations with zero or negative LivingArea or LotSizeArea
df = df[(df['LivingArea'] > 0) & (df['LotSizeArea'] > 0)]


In [70]:
# Remove observations where LivingArea > 0 but BedroomsTotal or BathroomsTotalInteger are 0
# or LivingArea == 0 but BedroomsTotal or BathroomsTotalInteger are > 0

df = df[
    ((df['LivingArea'] == 0) & (df['BedroomsTotal'] == 0) & (df['BathroomsTotalInteger'] == 0)) |
    ((df['LivingArea'] > 0) & (df['BedroomsTotal'] >= 1) & (df['BathroomsTotalInteger'] >= 1))
]

### Create train/test split

In [71]:
df["CloseDate"] = pd.to_datetime(df["CloseDate"])
df["Month"] = df["CloseDate"].dt.to_period("M")

months = sorted(df["Month"].unique())

X = 15

test_month = months[-1]
train_months = months[-(X + 1):-1]

train_df = df[df["Month"].isin(train_months)].copy()
test_df = df[df["Month"] == test_month].copy()

### Missing Value Handling on Training Data only:





In [72]:
# AttachedGarageYN column
train_df["AttachedGarageYN_missing"] = (
    train_df["AttachedGarageYN"].isna().astype(int)
)

test_df["AttachedGarageYN_missing"] = (
    test_df["AttachedGarageYN"].isna().astype(int)
)

garage_mode = train_df["AttachedGarageYN"].mode()[0]

train_df["AttachedGarageYN"] = (
    train_df["AttachedGarageYN"].fillna(garage_mode)
)

test_df["AttachedGarageYN"] = (
    test_df["AttachedGarageYN"].fillna(garage_mode)
)

/tmp/ipykernel_1233/1370537505.py:13: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df["AttachedGarageYN"].fillna(garage_mode)
/tmp/ipykernel_1233/1370537505.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_df["AttachedGarageYN"].fillna(garage_mode)


In [73]:
# numeric columns
# then, fill missing values with the Median
core_features = ["LivingArea", "LotSizeArea", "BedroomsTotal", "BathroomsTotalInteger"]

for col in core_features:

    train_df[col] = pd.to_numeric(
        train_df[col], errors="coerce"
    )

    test_df[col] = pd.to_numeric(
        test_df[col], errors="coerce"
    )

    median_value = train_df[col].median()

    train_df[col] = train_df[col].fillna(median_value)
    test_df[col] = test_df[col].fillna(median_value)


In [74]:
# replace missing values in the City and CountyOrParish columns with "Unknown"
train_df["City"] = train_df["City"].fillna("Unknown")
test_df["City"] = test_df["City"].fillna("Unknown")

train_df["CountyOrParish"] = train_df["CountyOrParish"].fillna("Unknown")
test_df["CountyOrParish"] = test_df["CountyOrParish"].fillna("Unknown")

In [75]:
# replace missing values of the YearBuilt column with median
# then, create a new column for age of the properties

train_df["YearBuilt_missing"] = (
    train_df["YearBuilt"].isna().astype(int)
)

test_df["YearBuilt_missing"] = (
    test_df["YearBuilt"].isna().astype(int)
)

year_median = train_df["YearBuilt"].median()

train_df["YearBuilt"] = train_df["YearBuilt"].fillna(year_median)
test_df["YearBuilt"] = test_df["YearBuilt"].fillna(year_median)

train_df["Age"] = 2026 - train_df["YearBuilt"]
test_df["Age"] = 2026 - test_df["YearBuilt"]

In [76]:
# replace missing values in Latitude and Longitude columns with median
lat_median = train_df["Latitude"].median()

train_df["Latitude"] = train_df["Latitude"].fillna(lat_median)
test_df["Latitude"] = test_df["Latitude"].fillna(lat_median)

long_median = train_df["Longitude"].median()

train_df["Longitude"] = train_df["Longitude"].fillna(long_median)
test_df["Longitude"] = test_df["Longitude"].fillna(long_median)

### Convert categorical fields to numeric (encoding)

In [77]:
# city frequency encoding
city_counts = train_df["City"].value_counts()

common_cities = city_counts[
    city_counts >= 50
].index


train_df["City"] = train_df["City"].where(
    train_df["City"].isin(common_cities),
    "Other"
)

test_df["City"] = test_df["City"].where(
    test_df["City"].isin(common_cities),
    "Other"
)

In [78]:
# One-hot encoding
train_df = pd.get_dummies(
    train_df,
    columns=["City", "CountyOrParish"],
    drop_first=True
)

test_df = pd.get_dummies(
    test_df,
    columns=["City", "CountyOrParish"],
    drop_first=True
)

In [79]:
# Make test columns exactly match train columns
# (important because test may miss some cities)
test_df = test_df.reindex(
    columns=train_df.columns,
    fill_value=0
)

In [80]:
X_train = train_df.drop(columns=["ClosePrice"])
y_train = train_df["ClosePrice"]

X_test = test_df.drop(columns=["ClosePrice"])
y_test = test_df["ClosePrice"]

### cleaned sets:

In [ ]:
train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)